# PRC Data Challenge 2026 — Exploración inicial de los datos

Cuaderno para observar las variables

In [2]:
# Setup
import pandas as pd

import config as C
import data

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 120)


---
## 1. La forma de los datos

Lo primero es saber con qué volumen y qué columnas trabajamos.

In [3]:
def dimensiones() -> pd.DataFrame:
    """Filas y columnas de cada uno de los tres conjuntos."""
    filas = []
    for tabla in ("training", "ranking", "submitting"):
        n = data.sql(f"SELECT count(*) AS n FROM {tabla}").iloc[0, 0]
        c = len(data.sql(f"DESCRIBE SELECT * FROM {tabla}"))
        filas.append({"tabla": tabla, "filas": int(n), "columnas": c})
    return pd.DataFrame(filas).set_index("tabla")


dimensiones()

,filas,columnas
tabla,,
training,4167797,30
ranking,689534,30
submitting,344841,2


In [4]:
def esquema(tabla: str = "training") -> pd.DataFrame:
    """Columnas, tipo, bloque de origen (_mvt / _flt) y un valor de ejemplo."""
    tipos = data.sql(f"DESCRIBE SELECT * FROM {tabla}")
    muestra = data.sql(f"SELECT * FROM {tabla} LIMIT 1")
    out = tipos[["column_name", "column_type"]].copy()
    out["bloque"] = out["column_name"].str.rsplit("_", n=1).str[-1]
    out["ejemplo"] = [muestra.iloc[0][c] if len(muestra) else None for c in out["column_name"]]
    return out.set_index("column_name")


esquema("training")

,column_type,bloque,ejemplo
column_name,,,
MVT_ID_mvt,DOUBLE,mvt,182234017.0
FLIGHT_ID_mvt,DOUBLE,mvt,280602939.0
FLIGHT_mvt,VARCHAR,mvt,LH982
FLIGHT_RULE_mvt,VARCHAR,mvt,I
ADEP_mvt,VARCHAR,mvt,EDDF
ADES_mvt,VARCHAR,mvt,EIDW
PHASE_mvt,VARCHAR,mvt,DEP
MVT_TIME_UTC_mvt,TIMESTAMP WITH TIME ZONE,mvt,2025-01-07 22:10:06+01:00
BLOCK_TIME_UTC_mvt,TIMESTAMP WITH TIME ZONE,mvt,2025-01-07 22:00:11+01:00


In [ ]:
# El esquema de la plantilla de envío
esquema("submitting")

,column_type,bloque,ejemplo
column_name,,,
MVT_ID_mvt,DOUBLE,mvt,202501153.0
TAXITIME_SEC_mvt,INTEGER,mvt,<NA>


In [6]:
def cabecera(tabla: str = "training", n: int = 4, phase: str | None = None) -> pd.DataFrame:
    """Primeras filas, traspuestas para ver las 30 columnas de un vistazo."""
    clause = f"WHERE PHASE_mvt = '{phase}'" if phase else ""
    return data.sql(f"SELECT * FROM {tabla} {clause} LIMIT {n}").T


# Una salida real de 2025: aquí SÍ están el off-block y el taxi-out.
cabecera("training", phase="DEP")

,0,1,2,3
MVT_ID_mvt,182234017.0,182234011.0,182234021.0,182234008.0
FLIGHT_ID_mvt,280602939.0,280602803.0,280602920.0,280602701.0
FLIGHT_mvt,LH982,LH796,LH958,LH778
FLIGHT_RULE_mvt,I,I,I,I
ADEP_mvt,EDDF,EDDF,EDDF,EDDF
ADES_mvt,EIDW,VHHH,EGBB,WSSS
PHASE_mvt,DEP,DEP,DEP,DEP
MVT_TIME_UTC_mvt,2025-01-07 22:10:06+01:00,2025-01-07 22:11:23+01:00,2025-01-07 22:12:29+01:00,2025-01-07 22:14:07+01:00
BLOCK_TIME_UTC_mvt,2025-01-07 22:00:11+01:00,2025-01-07 21:53:18+01:00,2025-01-07 22:01:01+01:00,2025-01-07 21:59:29+01:00
SCHED_TIME_UTC_mvt,2025-01-07 22:00:00+01:00,2025-01-07 21:50:00+01:00,2025-01-07 22:00:00+01:00,2025-01-07 21:45:00+01:00


**Qué mirar:** recorre la columna de arriba abajo y localiza la cadena física completa
de un despegue: `BLOCK_TIME_UTC_mvt` (off-block) → `MVT_TIME_UTC_mvt` (despegue), y
`TAXITIME_SEC_mvt` como la diferencia en segundos. 

`STAND_mvt` y `RUNWAY_mvt`: esos dos valores forman el **combo** sobre el
que EUROCONTROL calcula el tiempo de referencia.

In [7]:
# Una llegada de 2025: mismo esquema, pero aquí el taxi-time es taxi-IN.
cabecera("training", phase="ARR")

,0,1,2,3
MVT_ID_mvt,182249437.0,182249434.0,182249433.0,182249414.0
FLIGHT_ID_mvt,280601336.0,280602048.0,280601134.0,280601500.0
FLIGHT_mvt,LO383,LH121,BA916,LH947
FLIGHT_RULE_mvt,I,I,I,I
ADEP_mvt,EPWA,EDDM,EGLL,EGCC
ADES_mvt,EDDF,EDDF,EDDF,EDDF
PHASE_mvt,ARR,ARR,ARR,ARR
MVT_TIME_UTC_mvt,2025-01-07 22:10:32+01:00,2025-01-07 22:13:01+01:00,2025-01-07 22:15:50+01:00,2025-01-07 22:25:31+01:00
BLOCK_TIME_UTC_mvt,2025-01-07 22:18:57+01:00,2025-01-07 22:18:14+01:00,2025-01-07 22:24:49+01:00,2025-01-07 22:30:21+01:00
SCHED_TIME_UTC_mvt,2025-01-07 22:10:00+01:00,2025-01-07 22:05:00+01:00,2025-01-07 21:55:00+01:00,2025-01-07 20:30:00+01:00


In [8]:
# Y ahora la misma consulta sobre 2026. Compara con la salida de dos celdas más arriba.
cabecera("ranking", phase="DEP")

,0,1,2,3
MVT_ID_mvt,202501153.0,202501107.0,202501179.0,202501178.0
FLIGHT_ID_mvt,294702983.0,294703175.0,294703458.0,294703417.0
FLIGHT_mvt,RJ126,4Y1408,EN8806,LH584
FLIGHT_RULE_mvt,I,I,I,I
ADEP_mvt,EDDF,EDDF,EDDF,EDDF
ADES_mvt,OJAI,LBWN,LIML,HECA
PHASE_mvt,DEP,DEP,DEP,DEP
MVT_TIME_UTC_mvt,2026-01-04 15:55:26+01:00,2026-01-04 15:56:59+01:00,2026-01-04 16:00:00+01:00,2026-01-04 16:06:26+01:00
BLOCK_TIME_UTC_mvt,NaT,NaT,NaT,NaT
SCHED_TIME_UTC_mvt,2026-01-04 14:50:00+01:00,2026-01-04 13:25:00+01:00,2026-01-04 15:50:00+01:00,2026-01-04 15:50:00+01:00


`BLOCK_TIME_UTC_mvt` (únicamente en DEP) y `TAXITIME_SEC_mvt` aparecen vacíos, y `MVT_TIME_UTC_mvt` con dato.
 

---
## 3. ¿Qué hay dentro? Inventario de categóricas

Antes de decidir qué variables construir conviene saber cuántos valores distintos tiene
cada columna. La cardinalidad decide el tratamiento: `WK_TBL_CAT_flt` con 4 valores es
un one-hot trivial; `STAND_mvt` con miles necesita target encoding o agrupación por dique.

In [10]:
CATEGORICAS = [
    "ADEP_mvt", "ADES_mvt", "PHASE_mvt", "FLIGHT_RULE_mvt", "AIRCRAFT_TYPE_mvt",
    "RUNWAY_mvt", "STAND_mvt", "FLIGHT_mvt",
    "MARKET_SEGMENT_flt", "FLIGHT_TYPE_flt", "FLIGHT_RULE_flt", "AIRCRAFT_TYPE_flt",
    "WK_TBL_CAT_flt", "AIRCRAFT_OPERATOR_flt", "ADES_FILED_flt", "CALLSIGN_flt",
]


def cardinalidad(tabla: str = "training", cols: list[str] = CATEGORICAS) -> pd.DataFrame:
    """Valores distintos y cobertura de cada columna categórica."""
    sel = ", ".join(
        f'count(DISTINCT {c}) AS "{c}__n", '
        f'round(100.0 * count({c}) / count(*), 1) AS "{c}__cob"'
        for c in cols
    )
    s = data.sql(f"SELECT {sel} FROM {tabla}").iloc[0]
    return pd.DataFrame(
        {
            "valores_distintos": [int(s[f"{c}__n"]) for c in cols],
            "pct_no_nulo": [s[f"{c}__cob"] for c in cols],
        },
        index=cols,
    ).sort_values("valores_distintos", ascending=False)


cardinalidad("training")

,valores_distintos,pct_no_nulo
FLIGHT_mvt,74532,100.0
CALLSIGN_flt,73054,99.1
STAND_mvt,1923,100.0
ADEP_mvt,1581,100.0
ADES_mvt,1568,100.0
ADES_FILED_flt,1436,99.1
AIRCRAFT_OPERATOR_flt,682,99.1
AIRCRAFT_TYPE_mvt,271,100.0
AIRCRAFT_TYPE_flt,242,99.1
RUNWAY_mvt,58,100.0
